# Training PPO Tiny Tackers
Run on Google Colab for superior training efficiency
I recommend the T4 GPU

- - - - - - - - - - - - - - - - - - - - - - - -
## Train on gym_sail_steps SailboatReachRounding Environment

This notebook goes through the learning tasks in the same order required for the boat to complete a Triangle Racecourse. Each leg of that race demands that the PPO agent learn new skills.

The first learning task is to complete the environment built by Gabo Tor as this environment is the inspiration for the project. Subsequent learning tasks are all trained in environments created by yours truly. They are as follows:

    1. Start near the leeward buoy and reach the target point at the windward buoy
    2. Start near the windward buoy and reach the target point at the reach buoy
    3. Start near the reach buoy and reach the target point at the leeward buoy
    4. Start near the windward buoy and round the reach buoy finishing at the leeward buoy, all while chasing target points that are designed to encourage jibing
    5. Start near the windward buoy and reach the target point near the leeward buoy
    6. Hit every target point in a triangle race course by rounding all the buoys
- - - - - - - - - - - - - - - - - - - - - - - -



#####
#####
# *"Sailing is the art of going nowhere slowly, at great expense"*
*- The sailing community*


#####
#####
#####
#####

- - - - - - - - - - - - - - - - - - - -
# Training a PPO Baseline Model on Gabo-Tor Environment
- Learning Outcome: The Tiny Tacker must learn how to sail into the wind on a close haul while also learning to tack (turn through the wind without getting stuck in irons)
  1. Reach the Windward Buoy to Finish
- - - - - - - - - - - - - - - - - - - -

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Set directory to tiny_tackers
%cd "/content/drive/My Drive/tiny_tackers"

/content/drive/My Drive/tiny_tackers


In [3]:
# Install and import dependencies
!pip install stable-baselines3 gymnasium pygame

import os
import sys
import time
import pandas as pd
import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy

from gymnasium.wrappers import RecordVideo

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [4]:
# Set path to retrieving the base environment
BASE_ENV_PATH = "/content/drive/My Drive/tiny_tackers/gym_sailing_environments/gym_sailing_gabo-tor"
sys.path.append(BASE_ENV_PATH)

In [5]:
# Import environment and ensure it's comming from the correct path. Ending in tiny_tackers/gym_sailing_environments/gym_sailing_gabo-tor/gym_sailing/__init__.py
import gym_sailing
print(gym_sailing.__file__)

/content/drive/My Drive/tiny_tackers/gym_sailing_environments/gym_sailing_gabo-tor/gym_sailing/__init__.py


In [6]:
# Specifify that the environment is going to be one where continuous actions are possible
ENV_ID = "Sailboat-v0"

In [7]:
# Create directory for storing your model, videos, and metrics
models_dir = "models/ppo/base"
videos_dir = "videos/ppo/base"
metrics_dir = "metrics"

os.makedirs(models_dir, exist_ok=True)
os.makedirs(videos_dir, exist_ok=True)
os.makedirs(metrics_dir, exist_ok=True)

In [8]:
# Make and define training and evaluation environments (render mode is none because vizualizing significantly slows down training and evaluation)
def make_env(render_mode=None):
    env = gym.make(ENV_ID, render_mode=render_mode)
    env = Monitor(env)
    return env

train_env = make_env()
eval_env = make_env()

/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google.cloud')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-pa

In [9]:
# Improve efficiency with parallel environments
from stable_baselines3.common.env_util import make_vec_env
train_env = make_vec_env(lambda: make_env(), n_envs=4)

In [10]:
# Train PPO for 1_000_000 timesteps and save the training time in seconds
model = PPO(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=3e-4,
    n_steps=1024,
    batch_size=64,
    gamma=0.99,
    verbose=1,
)

start = time.time()
model.learn(
    total_timesteps=1_000_000,
    progress_bar=True,
)

training_time_seconds = time.time() - start

Using cuda device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: 
datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects 
to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Streaming output truncated to the last 5000 lines.
|    loss                 | 13.5         |
|    n_updates            | 150          |
|    policy_gradient_loss | -0.000748    |
|    std                  | 0.998        |
|    value_loss           | 36.6         |
------------------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 640          |
|    ep_rew_mean          | -230         |
| time/                   |              |
|    fps                  | 870          |
|    iterations           | 17           |
|    time_elapsed         | 80           |
|    total_timesteps      | 69632        |
| train/                  |              |
|    approx_kl            | 0.0026482742 |
|    clip_fraction        | 0.0229       |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.42        |
|    explained_variance   | 0.842        |
|    learning_rate        | 0.0003       |
|  

In [11]:
# Evaluate and capture average episode length and rewards over 20 episodes and view the results
episode_rewards, episode_lengths = evaluate_policy(
    model,
    eval_env,
    n_eval_episodes=20,
    deterministic=True,
    return_episode_rewards=True,
)

mean_reward = sum(episode_rewards) / len(episode_rewards)
std_reward = pd.Series(episode_rewards).std()

mean_episode_timesteps = sum(episode_lengths) / len(episode_lengths)
std_episode_timesteps = pd.Series(episode_lengths).std()

print("Mean reward:", mean_reward)
print("Std reward:", std_reward)
print("Mean episode timesteps:", mean_episode_timesteps)
print("Std episode timesteps:", std_episode_timesteps)
print("Training time seconds:", training_time_seconds)

Mean reward: 361.92136895
Std reward: 6.715062555944596
Mean episode timesteps: 1202.75
Std episode timesteps: 60.79722552533913
Training time seconds: 1177.0103106498718


In [12]:
# Save the base model if you are satisfied
model_path = f"{models_dir}/ppo_gabo-tor_base_1M"
model.save(model_path)

print(f"Saved model to: {model_path}.zip")

Saved model to: models/ppo/base/ppo_gabo-tor_base_1M.zip


In [13]:
# Save metrics if you are satisfied, or if you need to run this again and compare runs
metrics = pd.DataFrame([{
    "env_id": ENV_ID,
    "model": "PPO",
    "training_timesteps": 1_000_000,
    "n_eval_episodes": 20,
    "mean_reward": mean_reward,
    "std_reward": std_reward,
    "mean_episode_timesteps": mean_episode_timesteps,
    "std_episode_timesteps": std_episode_timesteps,
    "training_time_seconds": training_time_seconds,
}])

metrics_path = f"{metrics_dir}/ppo_base_metrics.csv"

if os.path.exists(metrics_path):
    old = pd.read_csv(metrics_path)
    metrics = pd.concat([old, metrics], ignore_index=True)

metrics.to_csv(metrics_path, index=False)
metrics

,env_id,model,training_timesteps,n_eval_episodes,mean_reward,std_reward,mean_episode_timesteps,std_episode_timesteps,training_time_seconds
0,Sailboat-v0,PPO,1000000,20,361.921369,6.715063,1202.75,60.797226,1177.010311


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Record video an evaluation episode of the trained model and save it in the videos directory with a name prefix of ppo_base_trained. The video should be saved as a mp4 file and should be named ppo_base_trained_episode_0.mp4.
video_env = gym.make(ENV_ID, render_mode="rgb_array")
video_env = RecordVideo(
    video_env,
    video_folder=videos_dir,
    name_prefix="ppo_base_trained",
    episode_trigger=lambda episode_id: episode_id == 0,
)

obs, info = video_env.reset()
done = False
truncated = False

while not (done or truncated):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = video_env.step(action)

import builtins # imported to fix bug with recording video
builtins.quit = lambda *args, **kwargs: None # imported to fix bug with recording video
video_env.close()


/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content/drive/MyDrive/tiny_tackers/videos/ppo/base folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Navigate to your videos folder and watch the video

---
---
---

- - - - - - - - - - - - - - - - - - - - - - - -
# Training on the gym_sail_steps Upwind Environment

- Learning Task: Similarly to the Gabo Tor environment, the agent must learn to sail from a leeward point to a target at the windward buoy
    1. The agent must learn to head towards the target (rounding point - green dot)
    2. The agent must learn how to mange points of sail and prevent getting stuck in "irons" (irons is when you are heading directly into the wind and begin to go backwards)
    3. The agent must optimize a path, you can make many or few turns. A good sailor tries to make as few turns as possible in order to maximize speed.
    4. All this must be managed while starting closer to the boundary of the environment because we are training for the traingle race.
- - - - - - - - - - - - - - - - - - - - - - - -

Note: Some of the code you will see below is redundant; it's there intentionally so that you may start at any point if you have a previously saved model.
For instance, you may not need to "mount" the drive in every section, but it's there in case you wanted to start at this point in the process.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Confirm my repo location
%cd "/content/drive/My Drive/tiny_tackers"
!pwd

In [ ]:
# Install Stable Baselines3 library if not already installed
!pip install stable-baselines3 gymnasium pygame

In [ ]:
# Confirm necessary libraries and imports are installed if they weren't already
import os
import sys
import time
import pandas as pd
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env
from gymnasium.wrappers import RecordVideo

In [ ]:
# Set Path to the gym_sail_steps environments
STEPS_ENV_PATH = "/content/drive/My Drive/tiny_tackers/gym_sailing_environments/gym_sail_steps"
sys.path.append(STEPS_ENV_PATH)

In [ ]:
# IMPORTANT! Import steps environments and ensure it's coming from the correct path. Ending in tiny_tackers/gym_sailing_environments/gym_sailing_race/gym_sail_race/__init__.py
import gym_sail_steps
print(gym_sail_steps.__file__)

In [ ]:
# Identify the upwind environment to call for this section
ENV_ID = "SailboatUpwind-v0"

In [ ]:
# Make folders for outcomes if folders do not already exist.
models_dir = "models/ppo/upwind_step"
videos_dir = "videos/ppo/upwind_step"
metrics_dir = "metrics"

os.makedirs(models_dir, exist_ok=True)
os.makedirs(videos_dir, exist_ok=True)
os.makedirs(metrics_dir, exist_ok=True)

In [ ]:
# Define the make environment function environments (render mode is none because visualizing significantly slows down training and evaluation)
def make_env(render_mode=None):
    env = gym.make(ENV_ID, render_mode=render_mode)
    env = Monitor(env)
    return env


In [ ]:
# Improve efficiency with parallel environments, create your training environment
from stable_baselines3.common.env_util import make_vec_env
train_env = make_vec_env(lambda: make_env(), n_envs=4)

In [ ]:
# Make evaluation environment
eval_env = make_env()

In [ ]:
# Train PPO for 1_000_000 timesteps and save the training time in seconds
# Hyperparameters inspired by
model = PPO(
    learning_rate=3e-4,
    n_steps=1024,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.0,
    vf_coef=0.5,
    max_grad_norm=0.5,
)

start = time.time()
model.learn(
    total_timesteps=1_000_000,
    progress_bar=True,
)

training_time_seconds = time.time() - start